# Dispenser 動作確認ノートブック

ディスペンサー（eco-PEN450 / Pololu Tic T500）を段階的に検証します。

| セクション | 内容 | 前提条件 |
|---|---|---|
| **1. コマンド単体テスト（Fake）** | `HighViscosityDispenserProprietaryFake` を直接呼び出してコマンドを確認 | なし |
| **2. アクションテスト（Fake）** | ノードに対して REST でアクションを呼び出し、ビジネスロジックを確認 | PC 上で Docker サービス + ノード起動 |
| **3. コマンド単体テスト（実機）** | `HighViscosityDispenserProprietary` で実機コマンドを確認 | 実機接続 + RPi 上で ser2net 起動済み |

接続先 IP・ポートは `devices/devices.settings.yaml` の `high_viscosity_dispenser.host` / `ser2net_port` で管理します。  
各セクションのセルがそれを自動で読み込むため、IP を変更する場合は YAML だけ編集すれば OK です。

---

**セクション 2 の起動コマンド（PC ターミナルで実行）:**
```powershell
cd "C:\Users\gussa\OneDrive - SEKISUI CHEMICAL CO.,LTD\ドキュメント\auto_calibration_lab"
docker compose up -d madsci_postgres madsci_mongodb event_manager resource_manager data_manager high_viscosity_liquid_weighing_1
```

> `node.settings.yaml` の `interface_type: fake` により、フェイクデバイスで動作します。

---
## セクション 1: コマンド単体テスト（Fake）

`HighViscosityDispenserProprietaryFake` を直接インスタンス化し、各コマンドを個別に確認します。  
**前提条件: なし**（サービス・実機不要）

In [21]:
import sys
import os
import importlib.util
from pathlib import Path

# プロジェクトルートの絶対パス
PROJECT_ROOT = Path(os.path.abspath(".."))

def import_device(filename: str):
    """devices/ ディレクトリのファイルを直接インポートする（__init__.py をスキップ）。"""
    path = PROJECT_ROOT / "devices" / filename
    module_name = path.stem
    spec = importlib.util.spec_from_file_location(module_name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

# Fake インターフェース（セクション 1）
_mod_fake = import_device("high_viscosity_dispenser_proprietary_fake.py")
HighViscosityDispenserProprietaryFake = _mod_fake.HighViscosityDispenserProprietaryFake

# 実インターフェース（セクション 3 で使用）
_mod_real = import_device("high_viscosity_dispenser_proprietary.py")
HighViscosityDispenserProprietary = _mod_real.HighViscosityDispenserProprietary

print("インポート成功")

インポート成功


In [22]:
# ---- 1-1. 初期化 ----
dispenser = HighViscosityDispenserProprietaryFake(
    port="FAKE",
    full_steps_per_rev=1036,   # devices.settings.yaml と同じ値
    microstep_multiplier=8,    # devices.settings.yaml と同じ値
    purge_speed_rps=0.5,       # purge 時の固定速度 [rev/s]（0.5 = 1.5 mL/min）
    latency=0.05,              # 実時間の 5% でシミュレーション（テスト高速化）
    failure_rate=0.0,          # エラー発生率（0.0 = エラーなし）
)
print(f"status: {dispenser.status}")
print(f"MAX_SPEED_ML_PER_MIN : {dispenser.MAX_SPEED_ML_PER_MIN} mL/min")
print(f"MIN_SPEED_ML_PER_MIN : {dispenser.MIN_SPEED_ML_PER_MIN} mL/min")
print(f"MIN_VOLUME_ML        : {dispenser.MIN_VOLUME_ML} mL")

status: connected
MAX_SPEED_ML_PER_MIN : 6.0 mL/min
MIN_SPEED_ML_PER_MIN : 0.5 mL/min
MIN_VOLUME_ML        : 0.004 mL


In [23]:
# ---- 1-2. purge（パージ / ノズル充填） ----
# ノズル内に液を充填する。初回使用前や気泡除去に使用。
PURGE_VOLUME_ML = 0.05  # パージ量 [mL]（1 rev 相当）

print(f"purge: {PURGE_VOLUME_ML} mL ...")
dispenser.purge(volume_ml=PURGE_VOLUME_ML)
print("purge 完了")

purge: 0.05 mL ...
purge 完了


In [24]:
# ---- 1-3. dispense（吐出） ----
# 指定量・指定速度で液を吐出する。
DISPENSE_VOLUME_ML = 0.05       # 吐出量 [mL]（1 rev 相当）
DISPENSE_SPEED_ML_PER_MIN = 2.0  # 吐出速度 [mL/min]（有効範囲: 0.5 〜 6.0）

print(f"dispense: {DISPENSE_VOLUME_ML} mL @ {DISPENSE_SPEED_ML_PER_MIN} mL/min ...")
dispenser.dispense(volume_ml=DISPENSE_VOLUME_ML, speed_ml_per_min=DISPENSE_SPEED_ML_PER_MIN)
print("dispense 完了")

dispense: 0.05 mL @ 2.0 mL/min ...
dispense 完了


In [25]:
# ---- 1-4. suck_back（サックバック / 液切り） ----
# 吐出後にわずかに逆回転させ、液だれを防ぐ。
# delay_s: 吐出完了からサックバック開始までの待機時間 [s]
#          Fake では latency 倍されたシミュレーション時間になる。
SUCK_BACK_DELAY_S = 0.5             # 待機時間 [s]
SUCK_BACK_VOLUME_ML = 0.01          # サックバック量 [mL]（dispense_volume より小さくすること）
SUCK_BACK_SPEED_ML_PER_MIN = 2.0   # サックバック速度 [mL/min]（有効範囲: 0.5 〜 6.0）

print(f"suck_back: delay={SUCK_BACK_DELAY_S}s, {SUCK_BACK_VOLUME_ML} mL @ {SUCK_BACK_SPEED_ML_PER_MIN} mL/min ...")
dispenser.suck_back(
    volume_ml=SUCK_BACK_VOLUME_ML,
    speed_ml_per_min=SUCK_BACK_SPEED_ML_PER_MIN,
    delay_s=SUCK_BACK_DELAY_S,
)
print("suck_back 完了")

suck_back: delay=0.5s, 0.01 mL @ 2.0 mL/min ...
suck_back 完了


In [26]:
# ---- 1-5. check_status ----
# ステータス確認（Fake では常に "connected"）
dispenser.check_status()
print(f"status: {dispenser.status}")

status: connected


In [27]:
# ---- 1-6. バリデーションエラーの確認 ----
# 範囲外パラメータを渡したときに ValueError が出ることを確認する。

import traceback

# (a) MIN_VOLUME_ML 未満
print("テスト (a): 最小量未満の dispense → ValueError 期待")
try:
    dispenser.dispense(volume_ml=0.001, speed_ml_per_min=1.0)
    print("  NG: 例外が発生しなかった")
except ValueError as e:
    print(f"  OK: {e}")

# (b) 速度の上限超過
print("テスト (b): MAX_SPEED_ML_PER_MIN 超過 → ValueError 期待")
try:
    dispenser.dispense(volume_ml=0.05, speed_ml_per_min=99.0)
    print("  NG: 例外が発生しなかった")
except ValueError as e:
    print(f"  OK: {e}")

# (c) 速度の下限未満
print("テスト (c): MIN_SPEED_ML_PER_MIN 未満 → ValueError 期待")
try:
    dispenser.dispense(volume_ml=0.05, speed_ml_per_min=0.1)
    print("  NG: 例外が発生しなかった")
except ValueError as e:
    print(f"  OK: {e}")

テスト (a): 最小量未満の dispense → ValueError 期待
  OK: rotations 0.02 is below minimum 0.08 rev (= MIN_VOLUME_ML 0.004 mL)
テスト (b): MAX_SPEED_ML_PER_MIN 超過 → ValueError 期待
  OK: speed_rps 32.99999999999999 is out of range [0.1667, 2.0] (= [0.5, 6.0] mL/min)
テスト (c): MIN_SPEED_ML_PER_MIN 未満 → ValueError 期待
  OK: speed_rps 0.03333333333333333 is out of range [0.1667, 2.0] (= [0.5, 6.0] mL/min)


In [28]:
# ---- 1-7. 切断 ----
dispenser.close()
print(f"status: {dispenser.status}")

status: disconnected


---
## セクション 2: アクションテスト（Fake）

ノードの REST API を通じてアクションを呼び出し、ビジネスロジック（Resource Manager 連携・Data Manager への保存など）を確認します。

Docker サービスとノードはどちらも **PC 上** で起動します。フェイクデバイスを使うため RPi 接続は不要です。

**手順:**

```powershell
cd "C:\Users\gussa\OneDrive - SEKISUI CHEMICAL CO.,LTD\ドキュメント\auto_calibration_lab"
docker compose up -d madsci_postgres madsci_mongodb event_manager resource_manager data_manager high_viscosity_liquid_weighing_1
```

> ノードが起動していない場合、接続セルはエラーになります。

In [39]:
# ---- 2-1. セットアップ ----
import yaml
import requests
from madsci.client.node.rest_node_client import RestNodeClient
from madsci.client.resource_client import ResourceClient
from madsci.common.types.action_types import ActionRequest
from madsci.common.types.client_types import RestNodeClientConfig

# セクション 2 はノードも Docker サービスも PC 上で動作するため、常に localhost を使用する
NODE_URL = "http://localhost:2000/"
RESOURCE_MANAGER_URL = "http://localhost:8003/"

# テスト用マテリアル名（セクション終了後に削除）
TEST_MATERIAL_NAME = "test1"

# 接続確認用に短いタイムアウト・リトライなし
_config = RestNodeClientConfig(timeout_default=5.0, retry_enabled=False, retry_total=0)

node_client = RestNodeClient(url=NODE_URL, config=_config)
resource_client = ResourceClient(resource_server_url=RESOURCE_MANAGER_URL)

print(f"node_client     : {NODE_URL}")
print(f"resource_client : {RESOURCE_MANAGER_URL}")
print()

# ノード接続確認（ノードが起動していない場合はエラーになります）
try:
    info = node_client.get_info()
    status = node_client.get_status()
    print(f"ノード名   : {info.node_name}")
    print(f"busy={status.busy}, paused={status.paused}, locked={status.locked}, errored={status.errored}")
except Exception as e:
    print(f"[ERROR] ノードに接続できません: {type(e).__name__}: {e}")
    print("  → PC 上でノードが起動しているか確認してください。")


2026-05-06T22:21:30.485831Z [info     ] EventClient initialized        client_name=madsci.client.node.rest_node_client event_server='Not configured' log_dir=.madsci\logs log_level=EventLogLevel.INFO madsci_version=0.7.0 platform=Windows-11-10.0.26200-SP0 python_version=3.13.2
2026-05-06T22:21:30.896152Z [info     ] EventClient initialized        client_name=madsci.client.resource_client event_server='Not configured' log_dir=.madsci\logs log_level=EventLogLevel.INFO madsci_version=0.7.0 platform=Windows-11-10.0.26200-SP0 python_version=3.13.2


node_client     : http://localhost:2000/
resource_client : http://localhost:8003/

ノード名   : high_viscosity_liquid_weighing_1
busy=False, paused=False, locked=False, errored=False


In [40]:
# ---- 2-2. テスト用マテリアルを登録 ----
# アクションは Resource Manager でのマテリアル存在確認を行うため、事前登録が必要。
# すでに存在する場合はスキップ。

from madsci.common.types.resource_types import Consumable
import requests

try:
    existing = resource_client.query_resource(resource_name=TEST_MATERIAL_NAME)
    print(f"マテリアルはすでに存在します: {TEST_MATERIAL_NAME}")
except requests.HTTPError as e:
    if e.response is not None and e.response.status_code == 404:
        material = Consumable(
            resource_name=TEST_MATERIAL_NAME,
            resource_class="high_viscosity_liquid",
            resource_description="dispenser_check.ipynb で作成したテスト用マテリアル",
            quantity=100.0,
            unit="mL",
            attributes={
                "schema_version": "0.0",
                "optimal_speed_ml_per_min": None,
                "g_per_rev": None,
                "density_g_per_cm3": None,
                "suck_back_delay_s": None,
                "suck_back_volume_ml": None,
                "suck_back_speed_ml_per_min": None,
            },
        )
        resource_client.add_resource(material)
        print(f"マテリアルを登録しました: {TEST_MATERIAL_NAME}")
    else:
        raise

マテリアルはすでに存在します: test1


In [41]:
# ---- 2-3. calibrate_dispenser アクション ----
# 速度を段階的に変えながら吐出質量を計測し、最適速度を求める。
# Fake モードではバランスのシミュレーション値が返る。

result = node_client.send_action(ActionRequest(
    action_name="calibrate_dispenser",
    args={
        "material_name": TEST_MATERIAL_NAME,
        "pressure_mpa": 0.3,               # 圧力（ラベルとして記録）
        "volume_per_step_ml": 0.05,         # 1 ステップの吐出量 [mL]
        "speed_start_ml_per_min": 1.0,      # 速度スイープ開始 [mL/min]
        "speed_end_ml_per_min": 3.0,        # 速度スイープ終了 [mL/min]
        "speed_step_ml_per_min": 1.0,       # 速度ステップ幅 [mL/min]
        "suck_back_volume_ml": 0.01,        # サックバック量 [mL]
        "suck_back_speed_ml_per_min": 2.0,  # サックバック速度 [mL/min]
    },
))

print(f"status: {result.status}")
if result.status == "succeeded":
    import json
    print(json.dumps(result.json_result, indent=2, ensure_ascii=False))
else:
    print(f"errors: {result.errors}")

status: ActionStatus.FAILED
errors: []


In [ ]:
# ---- 2-4. try_suck_back アクション ----
# 1 回だけ吐出 → 待機 → サックバックを行い、目視でパラメータを確認する。
# 実機接続時に繰り返し呼び出して最適なサックバック条件を探す際に使用。

result = node_client.send_action(ActionRequest(
    action_name="try_suck_back",
    args={
        "material_name": TEST_MATERIAL_NAME,
        "pressure_mpa": 0.3,
        "dispense_volume_ml": 0.05,          # 吐出量 [mL]
        "dispense_speed_ml_per_min": 2.0,     # 吐出速度 [mL/min]
        "suck_back_delay_s": 0.5,             # 吐出後の待機時間 [s]
        "suck_back_volume_ml": 0.01,          # サックバック量 [mL]
        "suck_back_speed_ml_per_min": 2.0,    # サックバック速度 [mL/min]
    },
))

print(f"status: {result.status}")
if result.status == "succeeded":
    import json
    print(json.dumps(result.json_result, indent=2, ensure_ascii=False))
else:
    print(f"errors: {result.errors}")

In [ ]:
# ---- 2-5. テスト用マテリアルを削除（クリーンアップ） ----
# テスト完了後に Resource Manager から削除する。

import requests

try:
    existing = resource_client.query_resource(resource_name=TEST_MATERIAL_NAME)
    resource_client.remove_resource(existing.resource_id)
    print(f"マテリアルを削除しました: {TEST_MATERIAL_NAME}")
except requests.HTTPError as e:
    if e.response is not None and e.response.status_code == 404:
        print(f"マテリアルは存在しませんでした（スキップ）")
    else:
        raise

---
## セクション 3: コマンド単体テスト（実機）

`HighViscosityDispenserProprietary` を直接インスタンス化し、実機に対してコマンドを確認します。

**前提条件:**
- ディスペンサーが RPi の GPIO14/15 (UART) に接続済み
- RPi 上で ser2net が起動していること（下記参照）
- Tic Control Center で以下を設定済み:
  - Control mode: **Serial / I2C / USB**
  - Command timeout: **disabled**
  - Baud rate: **9600**
  - Step mode: `microstep_multiplier` と一致（デフォルト: 1/8 step）

接続先は `devices/devices.settings.yaml` の `host` / `ser2net_port` から自動で読み込まれます。

**RPi 上での ser2net セットアップ（初回のみ）:**
```bash
# 1. UART 有効化（要再起動）
sudo raspi-config  # Interface Options → Serial Port → login shell: No, hardware enabled: Yes

# 2. ser2net インストール
sudo apt install ser2net

# 3. /etc/ser2net.yaml を編集して以下を追記
#   connection: &serial0
#     accepter: tcp,2217
#     connector: serialdev,/dev/serial0,9600n81,local
#     options:
#       kickolduser: true

# 4. ser2net を自動起動設定 + 開始
sudo systemctl enable --now ser2net
```

> **注意**: 実機を動かすため、ノズル先端・吐出先の安全を確認してから実行してください。

In [ ]:
# ---- 3-1. 利用可能な COM ポートを確認 ----
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
if ports:
    print("検出された COM ポート:")
    for p in ports:
        print(f"  {p.device:10s}  {p.description}")
else:
    print("COM ポートが見つかりません（実機未接続）")

In [ ]:
# ---- 3-2. 実機の初期化 ----
# 接続先は devices/devices.settings.yaml から自動で読み込まれます。
# host が設定されている場合は RFC2217 (ser2net) 経由、null の場合はローカルポート直接接続。
import yaml

_devices_cfg = yaml.safe_load(
    (PROJECT_ROOT / "devices" / "devices.settings.yaml").read_text()
)
_dcfg = _devices_cfg["high_viscosity_dispenser"]

# ↓ 実機で確認した安全な purge 速度 [rev/s] を設定（最大 2.0）
PURGE_SPEED_RPS = 0.5  # 0.5 rev/s = 1.5 mL/min

dispenser_real = HighViscosityDispenserProprietary(
    host=_dcfg.get("host"),                        # Tailscale IP（None = ローカル直接接続）
    ser2net_port=_dcfg.get("ser2net_port", 2217),
    port=_dcfg["port"],
    full_steps_per_rev=_dcfg["full_steps_per_rev"],
    microstep_multiplier=_dcfg["microstep_multiplier"],
    purge_speed_rps=PURGE_SPEED_RPS,
    baud_rate=_dcfg["baud_rate"],
)
print(f"status: {dispenser_real.status}")

In [ ]:
# ---- 3-3. purge（パージ）----
# ノズル先端が容器の上にあることを確認してから実行してください。

PURGE_VOLUME_ML = 0.05  # パージ量 [mL]

print(f"purge: {PURGE_VOLUME_ML} mL ...")
dispenser_real.purge(volume_ml=PURGE_VOLUME_ML)
print("purge 完了")

In [ ]:
# ---- 3-4. dispense（吐出） ----
DISPENSE_VOLUME_ML = 0.05
DISPENSE_SPEED_ML_PER_MIN = 2.0

print(f"dispense: {DISPENSE_VOLUME_ML} mL @ {DISPENSE_SPEED_ML_PER_MIN} mL/min ...")
dispenser_real.dispense(volume_ml=DISPENSE_VOLUME_ML, speed_ml_per_min=DISPENSE_SPEED_ML_PER_MIN)
print("dispense 完了")

In [ ]:
# ---- 3-5. suck_back（サックバック） ----
SUCK_BACK_DELAY_S = 0.5
SUCK_BACK_VOLUME_ML = 0.01
SUCK_BACK_SPEED_ML_PER_MIN = 2.0

print(f"suck_back: delay={SUCK_BACK_DELAY_S}s, {SUCK_BACK_VOLUME_ML} mL @ {SUCK_BACK_SPEED_ML_PER_MIN} mL/min ...")
dispenser_real.suck_back(
    volume_ml=SUCK_BACK_VOLUME_ML,
    speed_ml_per_min=SUCK_BACK_SPEED_ML_PER_MIN,
    delay_s=SUCK_BACK_DELAY_S,
)
print("suck_back 完了")

In [ ]:
# ---- 3-6. start_rotation / stop_rotation（連続回転）----
# 連続的に回転を開始し、手動で stop_rotation() を呼ぶまで回り続ける。
# 流量の目視確認や、長時間連続吐出のテストに使用。
import time

START_SPEED_RPS = 0.5   # 回転速度 [rev/s]（0.5 = 1.5 mL/min）
DIRECTION = +1          # +1: 吐出方向, -1: サックバック方向
RUN_DURATION_S = 3.0    # 連続回転時間 [s]（テスト用）

print(f"start_rotation: {START_SPEED_RPS} rev/s, direction={DIRECTION}")
dispenser_real.start_rotation(speed_rps=START_SPEED_RPS, direction=DIRECTION)
time.sleep(RUN_DURATION_S)
dispenser_real.stop_rotation()
print(f"{RUN_DURATION_S} 秒後に停止完了")

In [ ]:
# ---- 3-7. check_status ----
dispenser_real.check_status()
print(f"status: {dispenser_real.status}")

In [ ]:
# ---- 3-8. 切断 ----
# モーターを安全停止してシリアル接続を閉じる。
# テスト完了後は必ず実行してください。

dispenser_real.close()
print(f"status: {dispenser_real.status}")